In [ ]:
# Librerie per la soluzione del sistema lineare
import numpy as np
from scipy.sparse import coo_matrix, csc_matrix
from scipy.sparse.linalg import spsolve_triangular
from sksparse.cholmod import cholesky

# Costruzione dei grafici
import matplotlib.pyplot as plt

# N = sqrt(punti del dominio)
N = [128]

N_plot = 128

In [ ]:
def sistema_lineare(N, ord):
    """ N**2 = numero di punti del dominio
        ordinamento = naturale (inserire "old") o nuovo (inserire "new") per distinguere le due tipologie di file
        Restituisce la matrice a del sistema lineare in formato CSC e il vettore b dei termini noti
    """
    termine_noto = f"output/rhs_{N}_{ord}.txt" 
    matrice_sistema = f"output/A_{N}_{ord}.txt"
    
    b = np.loadtxt(termine_noto)    # vettore dei termini noti

    a = np.loadtxt(matrice_sistema) # matrice del sistema lineare
    
    # SciPy, per costruire una matrice sparsa, pretende che gli indici siano numeri interi! (np.loadtxt --> float64)
    prima_colonna = a[:, 0].astype(int)     # i
    seconda_colonna = a[:, 1].astype(int)   # j
    terza_colonna = a[:, 2]                 # a[i, j], ingressi a
    # Creazione a COO (i dati devono essere le colonne separate lette dal file)
    a_coo = coo_matrix((terza_colonna, (prima_colonna, seconda_colonna)), shape=(N**2, N**2))

    # Conversione in CSC
    a_csc = a_coo.tocsc()

    return a_csc, b

def my_Cholesky(a):
    factor = cholesky(a, order="natural")

    return csc_matrix(factor[0])

def soluzione_sistemaLineare(l, b):
    """ Ax = (LL^T)x = b, con L triangolare inferiore
        Ly = b --> ottengo y per sostituzione in avanti
        Lx = y --> ottengo x per sostituzione all'indietro
    """
    y = spsolve_triangular(l, b, lower=True)    # sostituzione in avanti
    x = spsolve_triangular(l.T, y, lower=False) # sostituzione all'indietro
    
    return x

In [ ]:
# Soluzioni del sistema
x = {}

for k in N: # k = chiave dei dizionari
    # --- Nuovo ordinamento ---
    a, b = sistema_lineare(k, "new")

    l = my_Cholesky(- a)

    soluzione = soluzione_sistemaLineare(l, - b)
    
    x[k] = soluzione

In [ ]:
# Ridimensiono il vettore (N_plot^2) in una matrice N_plot x N_plot
soluzione_2d = x[N_plot].reshape((N_plot, N_plot))

fig, axs = plt.subplots()

axs.set(title=f"Soluzione Equazione del Calore (N={N_plot})", xlabel='x', ylabel='y')

# Soluzione
mappa_calore = axs.contourf(soluzione_2d, cmap='hot', levels=50, extent=[0, 1, 0, 1])
fig.colorbar(mappa_calore, ax=axs, label='Temperatura')

plt.show()